# Learning PyTorch with examples

This tutorial introduces the fundamental concepts of PyTorch through self-contained examples.

At its core, PyTorch provides two main features:

- An n-dimensional Tensor, similar to numpy but can run on GPUs
- Automatic differentiation for building and training neural networks

We will use a fully-connected ReLU network as our running example. The network will have a single hidden layer, and will be trained with gradient descent to fit random data by minimizing the Euclidean distance between the network output and the true output.

## Warm-up: numpy

Before introducing PyTorch, we will first implement the network using numpy.

Numpy provides an n-dimensional array object, and many functions for manipulating these arrays. Numpy is a generic framework for scientific computing; it does not know anything about computation graphs, or deep learning, or gradients. However we can easily use numpy to fit a two-layer network to random data by manually implementing the forward and backward passes through the network using numpy operations:

In [1]:
import numpy as np

# N is batch size, D_in is input dimension, H is hidden dimension, D_out is ouput dimension
N = 64
D_in = 1000
H = 100
D_out = 10

# create random input and output data
x = np.random.randn(N, D_in)
y = np.random.randn(N, D_out)

# randomly initialize weights
w1 = np.random.randn(D_in, H)
w2 = np.random.randn(H, D_out)

# define learning rate
learning_rate = 1e-6
for t in range(50):
    # forward pass: compute predicted y
    h = x.dot(w1)
    h_relu = np.maximum(h, 0)
    y_pred = h_relu.dot(w2)
    
    # compute and print loss
    loss = np.square(y_pred - y).sum()
    print(t, loss)
    
    # backprop to compute gradients of w1 and w2 with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_w2 = h_relu.T.dot(grad_y_pred)
    grad_h_relu = grad_y_pred.dot(w2.T)
    grad_h = grad_h_relu.copy()
    grad_h[h < 0] = 0
    grad_w1 = x.T.dot(grad_h)
    
    # update weights
    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2

0 30052674.563473135
1 23920684.484657474
2 21549324.962169077
3 19640921.813898765
4 16973617.10897804
5 13373910.182778984
6 9662860.275767904
7 6509922.725545971
8 4245698.804865476
9 2763843.9952351376
10 1846995.3410865623
11 1285579.0265265445
12 937680.1097362882
13 714963.1899836354
14 565838.5543163
15 460901.332299893
16 383639.44871506776
17 324404.1414866833
18 277376.8491898139
19 239230.3900008152
20 207704.80724055093
21 181335.61674680925
22 158986.99554407425
23 139922.31980083635
24 123555.54961280624
25 109445.90874766199
26 97188.82481748748
27 86510.63867169485
28 77175.1306535296
29 68986.2268683532
30 61780.31291946801
31 55423.833258651684
32 49803.849000616894
33 44824.38691514191
34 40405.41902955764
35 36474.368628600874
36 32971.46120118189
37 29841.422320620328
38 27041.506789011357
39 24532.637715029883
40 22281.018063075782
41 20257.942974358466
42 18436.774965873497
43 16794.895610167772
44 15313.387767666789
45 13975.063270261755
46 12764.217670830118
4

## PyTorch: Tensors

Numpy is a great framework, but it cannot utilize GPUs to accelerate its numerical computations. For modern deep neural networks, GPUs often provide speedups of 50x or greater, so unfortunately numpy won’t be enough for modern deep learning.

Here we introduce the most fundamental PyTorch concept: the Tensor. A PyTorch Tensor is conceptually identical to a numpy array: a Tensor is an n-dimensional array, and PyTorch provides many functions for operating on these Tensors. Behind the scenes, Tensors can keep track of a computational graph and gradients, but they’re also useful as a generic tool for scientific computing.

Also unlike numpy, PyTorch Tensors can utilize GPUs to accelerate their numeric computations. To run a PyTorch Tensor on GPU, you simply need to cast it to a new datatype.

Here we use PyTorch Tensors to fit a two-layer network to random data. Like the numpy example above we need to manually implement the forward and backward passes through the network:

In [16]:
import torch

dtype = torch.float
device = torch.device('cpu')
# device = torch.device('cuda:0') # uncomment this to run on GPU

# N is batch size, D_in is unputs dimension, H is hidden dimension, D_out is output dimension
N, D_in, H, D_out = 64, 1000, 100, 10

# create random input and output data
x = torch.randn(N, D_in, device = device, dtype = dtype)
y = torch.randn(N, D_out, device = device, dtype = dtype)

# randomly initialize weights
w1 = torch.randn(D_in, H, device = device, dtype = dtype)
w2 = torch.randn(H, D_out, device = device, dtype = dtype)

# define learning rate
learning_rate = 1e-6
for t in range(500):
    # forward pass: compute predicted y
    h = x.mm(w1)
    h_relu = h.clamp(min = 0)
    y_pred = h_relu.mm(w2)
    
    # compute and print loss
    loss = (y_pred - y).sum().item()
    if t % 100 == 99:
        print(t, loss)
        
    # backdrop to compute gradients of w1 and w2 with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_w2 = h_relu.t().mm(grad_y_pred)
    grad_h_relu = grad_y_pred.mm(w2.t())
    grad_h = grad_h_relu.clone() 
    grad_h[h < 0] = 0
    grad_w1 = x.t().mm(grad_h)
    
    # update weights using gradient descent
    w1 -= learning_rate * grad_w1
    w2 -= learning_rate * grad_w2

99 -4.479207515716553
199 -0.20482437312602997
299 -0.010451012291014194
399 -0.001206742599606514
499 -0.00105181522667408


## Autograd
## PyTorch: Tensors and autograd

In the above examples, we had to manually implement both the forward and backward passes of our neural network. Manually implementing the backward pass is not a big deal for a small two-layer network, but can quickly get very hairy for large complex networks.

Thankfully, we can use automatic differentiation to automate the computation of backward passes in neural networks. The autograd package in PyTorch provides exactly this functionality. When using autograd, the forward pass of your network will define a computational graph; nodes in the graph will be Tensors, and edges will be functions that produce output Tensors from input Tensors. Backpropagating through this graph then allows you to easily compute gradients.

This sounds complicated, it’s pretty simple to use in practice. Each Tensor represents a node in a computational graph. If x is a Tensor that has x.requires_grad=True then x.grad is another Tensor holding the gradient of x with respect to some scalar value.

Here we use PyTorch Tensors and autograd to implement our two-layer network; now we no longer need to manually implement the backward pass through the network:

In [15]:
import torch

dtype = torch.float
device = torch.device('cpu')
# device = torch.device('cuda:0') # uncomment this to run on GPU

# N is batch size, D_in is input dimension, H is hidden dimension, D_out is ouput dimension
N, D_in, H, D_out = 64, 1000, 100, 10

# create random tensors to hold input and output
# setting requires_grad = Flase indicates that we do not need to compute gradients
# with respect to these tensors during the backward pass
x = torch.randn(N, D_in, device = device, dtype = dtype)
y = torch.randn(N, D_out, device = device, dtype = dtype)

# create random tensors for weights
# setting requires_grad = True indicates that we want to compute gradients
# with respect to these tensors during backward pass
w1 = torch.randn(D_in, H, device = device, dtype = dtype, requires_grad = True)
w2 = torch.randn(H, D_out, device = device, dtype = dtype, requires_grad = True)

# define learning rate
learning_rate = 1e-6
for t in range(500):
    # forward pass: compute predicted y using operations on tensors
    # these are exactly the same operations we used to compute the forward pass using tensors
    # but we do not need to keep references to intremediate values
    # since we are not implementing the backward pass by hand
    y_pred = x.mm(w1).clamp(min = 0).mm(w2)
    
    # compute and print loss using operations on tensors
    # now, loss is a tensor os shape (1,)
    # loss.item() gets the scalar value held in the loss
    loss = (y_pred - y).pow(2).sum()
    if t % 100 == 99:
        print(t, loss.item())
        
    # use autograd to compute the backward pass
    # this call will compute the gradient of loss with respect to all tensors with requires_grad = True
    # after this call w1.grad and w2.grad will be tensors holding the gradient of the loss with respect to w1 and w2 respectively
    loss.backward()
    
    # manually update weights using gradient descent
    # wrap it in torch.no_grad() because weights have requires_grad = True
    # but we do not need to track this in autograd
    # an alternative way is to operate on weight.data and weight.grad.data
    # recall that tensor.data gives a tensor that shares the storage with tensor
    # but does not track history
    # we can also use torch.optim.SGD to achieve this
    with torch.no_grad():
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad
        
        # manually zero the gradients after updating weights
        w1.grad.zero_()
        w2.grad.zero_()

99 541.2438354492188
199 1.9724007844924927
299 0.011218567378818989
399 0.00021793889754917473
499 3.222507439204492e-05


## PyTorch: Defining new autograd functions

Under the hood, each primitive autograd operator is really two functions that operate on Tensors. The forward function computes output Tensors from input Tensors. The backward function receives the gradient of the output Tensors with respect to some scalar value, and computes the gradient of the input Tensors with respect to that same scalar value.

In PyTorch we can easily define our own autograd operator by defining a subclass of torch.autograd.Function and implementing the forward and backward functions. We can then use our new autograd operator by constructing an instance and calling it like a function, passing Tensors containing input data.

In this example we define our own custom autograd function for performing the ReLU nonlinearity, and use it to implement our two-layer network:

In [14]:
import torch

class MyReLU(torch.autograd.Function):
    # we can implement our own custom autograd Functions by subclassing torch.autograd.Function
    # and implementing the forward and backward passes which operate on tensors
    
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return input.clamp(min = 0)
        # in the forward pass we receive a tensor containing the input
        # and return a tensor a tensor containing the output
        # ctx is a context object that can be used to stash information for backward propagation
        # we can cache arbitrary objects for use in the backward pass using ctx.save_for_backward method
        
    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad_input[input < 0] = 0
        return grad_input
        # in the backward pass we receive a tensor containing the gradient of the loss with respect to the output
        # and we need to compute the gradient of the loss with respect to the input
        
dtype = torch.float
device = torch.device('cpu')
# device = torch.device('cuda:0') # uncomment this to run on GPU

# N is batch size, D_in is input dimension, H is hidden dimension, D_out is output dimension
N, D_in, H, D_out = 64, 1000, 100, 10

# create random tensors to hold input and output
x = torch.randn(N, D_in, device = device, dtype = dtype)
y = torch.randn(N, D_out, device = device, dtype = dtype)

# create random tensors for weights
w1 = torch.randn(D_in, H, device = device, dtype = dtype, requires_grad = True)
w2 = torch.randn(H, D_out, device = device, dtype = dtype, requires_grad = True)

# define learning rate
learning_rate = 1e-6
for t in range(500):
    # to apply our Function, we use Function.apply method, we alias this as relu
    relu = MyReLU.apply
    
    # forward pass: compute predicted y using operations
    # we compute ReLU using our custom autograd operation
    y_pred = relu(x.mm(w1)).mm(w2)
    
    # compute and print loss 
    loss = (y_pred - y).pow(2).sum()
    if t % 100 == 99:
        print(t, loss.item())
        
    # use autograd to compute the backward pass 
    loss.backward()
    
    # update weights using gradient descent
    with torch.no_grad():
        w1 -= learning_rate * w1.grad
        w2 -= learning_rate * w2.grad
        
        # manually zero the gradients after updating weights
        w1.grad.zero_()
        w2.grad.zero_()

99 553.8711547851562
199 3.03348445892334
299 0.027317728847265244
399 0.0005070215556770563
499 6.252516323002055e-05


## nn module
## PyTorch: nn

Computational graphs and autograd are a very powerful paradigm for defining complex operators and automatically taking derivatives; however for large neural networks raw autograd can be a bit too low-level.

When building neural networks we frequently think of arranging the computation into layers, some of which have learnable parameters which will be optimized during learning.

In TensorFlow, packages like Keras, TensorFlow-Slim, and TFLearn provide higher-level abstractions over raw computational graphs that are useful for building neural networks.

In PyTorch, the nn package serves this same purpose. The nn package defines a set of Modules, which are roughly equivalent to neural network layers. A Module receives input Tensors and computes output Tensors, but may also hold internal state such as Tensors containing learnable parameters. The nn package also defines a set of useful loss functions that are commonly used when training neural networks.

In this example we use the nn package to implement our two-layer network:

In [13]:
import torch

# N is batch size, D_in is input dimension, H is hidden dimension, D_out is output dimension
N, D_in, H, D_out = 64, 1000, 100, 10

# create random tensors to hold input and output
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)

# use the nn package to define our model as a sequence of layers
# nn.Sequential is a module which contains other modules
# and applies them in sequence to produce its output
# each linear module computes output from input using a linear function
# and holds internal tensors for its weigth and bias
model = torch.nn.Sequential(torch.nn.Linear(D_in, H),
                            torch.nn.ReLU(),
                            torch.nn.Linear(H, D_out))

# the nn package also contains definitions of popular loss functions
# in this case we will use mean square error (MSE) as our loss function
loss_fn = torch.nn.MSELoss(reduction = 'sum')

# define learning rate
learning_rate = 1e-4
for t in range(500):
    # forward pass: compute predicted y by passing x to the model
    # module objetcs override the __call__ operator so we can call them like functions
    # when doing so you pass a tensor of input data to the module and produces a tensor of output data
    y_pred = model(x)
    
    # compute and print loss: we pass tensors containg the predicted and true values of y
    # and the loss function returns a tensor containing the loss
    loss = loss_fn(y_pred, y)
    if t % 10 == 99:
        print(t, loss.item())
        
    # zero the gradients before running the backward pass
    model.zero_grad()
    
    # backward pass: compute gradient of the loss with repsect to all learnable parameters of the model
    # internally, the parameters of each module are stored in tensors with requires_grad = True
    # so this call will compute gradients for all learnable parameters in the model
    loss.backward()
    
    # update the weights using gradient descent
    # each parameter is a tensor, so we can access its gradients like we did before
    with torch.no_grad():
        for param in model.parameters():
            param -= learning_rate * param.grad

## PyTorch: optim

Up to this point we have updated the weights of our models by manually mutating the Tensors holding learnable parameters (with torch.no_grad() or .data to avoid tracking history in autograd). This is not a huge burden for simple optimization algorithms like stochastic gradient descent, but in practice we often train neural networks using more sophisticated optimizers like AdaGrad, RMSProp, Adam, etc.

The optim package in PyTorch abstracts the idea of an optimization algorithm and provides implementations of commonly used optimization algorithms.

In this example we will use the nn package to define our model as before, but we will optimize the model using the Adam algorithm provided by the optim package:

In [11]:
import torch

# N is batch size, D_in is input dimesnsion, H is hidden dimension, D_out is output dimension
N, D_in, H, D_out = 64, 1000, 100, 10

# creare random tensors to hold input and output
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)

# use the nn package to define our model and loss function
model = torch.nn.Sequential(torch.nn.Linear(D_in, H),
                            torch.nn.ReLU(),
                            torch.nn.Linear(H, D_out))
loss_fn = torch.nn.MSELoss(reduction = 'sum')

# use the optim package to define an optimizer that will update the weights of the model
# we will use Adam, the optim package containsmany other optimization algorithms
# the first argument to the Adam constructor tells the optimizer which tensors it should update
learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
for t in range(500):
    # forward pass: compute predicted y by passing x to the model
    y_pred = model(x)
    
    # compute and print loss
    loss = loss_fn(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())
        
    # before backaward pass, use the optimizer object to zero all of the gradients for variables it will update
    # which are the learnable weights of the model
    # this is because by default, graduents are accumulated in buffers(i.e. not overwritten) whenever .backward() is called
    # ckeckout docs for torch.autograd.backward for more details
    optimizer.zero_grad()
    
    # backward pass: compute gradient of the loss with respect to model parameters
    loss.backward()
    
    # calling the step function on an optimizer makes an update to its parameters
    optimizer.step()

99 44.78773498535156
199 0.6135339140892029
299 0.006967842113226652
399 4.255479507264681e-05
499 3.827680927770416e-07


## PyTorch: Custom nn Modules

Sometimes you will want to specify models that are more complex than a sequence of existing Modules; for these cases you can define your own Modules by subclassing nn.Module and defining a forward which receives input Tensors and produces output Tensors using other modules or other autograd operations on Tensors.

In this example we implement our two-layer network as a custom Module subclass:

In [10]:
import torch

class TwoLayerNet(torch.nn.Module):
    def __init__(self, D_in, H, D_out):
        # in the constructor we instantiate two nn.Linear modules
        # and assign them as memeber variables
        super(TwoLayerNet, self).__init__()
        self.linear1 = torch.nn.Linear(D_in, H)
        self.linear2 = torch.nn.Linear(H, D_out)
        
    def forward(self, x):
        # in the forward function we accept a tensor of input data
        # and we must return a tensor of output data
        # we can use modules defined in the constructor 
        # as well as arbitrary operators on tensors
        h_relu = self.Linear1(x).clamp(min = 0)
        y_pred = self.linear2(h_relu)
        return y_pred
    
# N is batch size, D_in is input dimension, H is hidden dimension, D_out is output dimension
N, D_in, H, D_out = 64, 1000, 100, 10

# create random tensors to hold input and output
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)

# construct our loss function and an optimizer
# to call model.parameters() in SGD constructor will contain the larnable parameters of the two nn.Linear modules
# which are members of the model
criterion = torch.nn.MSELoss(reduction = 'sum')
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-4)
for t in range(500):
    # forward pass: compute predicted y by passing x to the model
    y_pred = model(x)
    
    # compute and print loss
    loss = criterion(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())
        
    # zero gradients, perform a backward pass and update the weights
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

99 1.943150520324707
199 0.02542617917060852
299 0.0005824005929753184
399 1.6158865037141368e-05
499 4.980572612112155e-07


## PyTorch: Control Flow + Weight Sharing

As an example of dynamic graphs and weight sharing, we implement a very strange model: a fully-connected ReLU network that on each forward pass chooses a random number between 1 and 4 and uses that many hidden layers, reusing the same weights multiple times to compute the innermost hidden layers.

For this model we can use normal Python flow control to implement the loop, and we can implement weight sharing among the innermost layers by simply reusing the same Module multiple times when defining the forward pass.

We can easily implement this model as a Module subclass:

In [17]:
import random
import torch

class DynamicNet(torch.nn.Module):
    def __init__(self, D_in, H, D_out):
        # in the constructor we construct three nn.Linear instatnces
        # that we will use in the forward pass
        super(DynamicNet, self).__init__()
        self.input_linear = torch.nn.Linear(D_in, H)
        self.middle_linear = torch.nn.Linear(H, H)
        self.output_linear = torch.nn.Linear(H, D_out)
        
    def forward(self, x):
        # for the forward pass of the model, we randomly choose either 0, 1, 2, or 3
        # and reuse the the middle_linear module that many times to compute hidden layer representations
        # since each forward pass builds a dynamic computation graph
        # we can use normal Python control flow operators like loops or conditional statements
        # when defining the forward pass of the model
        # here we also see that it is perfectly safe to reuse the same module many times
        # when defining a computational graph
        # this is a big improvement from Lua Torch, where each module could be used only once
        h_relu = self.input_linear(x).clamp(min = 0)
        for _ in range(random.randint(0, 3)):
            h_relu = self.middle_linear(h_relu).clamp(min = 0)
        y_pred = self.output_linear(h_relu)
        return y_pred
    
# N is batch size, D_in is input dimension, H is hidden dimension, D_out is output dimension
N, D_in, H, D_out = 64, 1000, 100, 10

# creare random tensors to hold input and output
x = torch.randn(N, D_in)
y = torch.randn(N, D_out)

# construct the model by instantiating the class defined above
model = DynamicNet(D_in, H, D_out)

# construct loss function and optimizer
# training this strange model with vanilla stochastic gradient descent is tough
# so we use momentum
criterion = torch.nn.MSELoss(reduction = 'sum')
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-4, momentum = 0.9)
for t in range(500):
    # forward pass: compute predicted y by passing x to the model
    y_pred = model(x)
    
    # compute and print loss
    loss = criterion(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())
        
    # zero gradients, perform a backward pass and update the weights
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

99 25.49372100830078
199 2.3700742721557617
299 6.150790214538574
399 0.41677993535995483
499 0.08236284554004669
